# Module 5: Regulation Movements Staging

This notebook creates the regulation movement staging tables in `main.regtech_ops_stg`.

**Replicates SSIS package:** `Regulation_Movments_Report.dtsx` (truncate/reload)

**Targets created:**
- `bi_output_regtechops_reg_migrationinout_population` — Migration population snapshot from certified gold
- `bi_output_regtechops_reg_regulationinoutdailydata` — Regulation in/out daily data from certified gold
- `bi_output_regtechops_reg_regulation_movments_positions` — Movement positions (open + closed + migration-only)

**Dependencies:** Module 4A (split-price data for EOD enrichment)

In [0]:
dbutils.widgets.text("report_date", "2026-06-09", "Report Date (YYYY-MM-DD)")

report_date = dbutils.widgets.get("report_date")
print(f"Running Module 5: Regulation Movements Staging for report_date = {report_date}")

In [0]:
%sql
-- Reg_MigrationInOut_Population: derived from snapshot regulationinoutdailydata
-- Original source (STALE since 2023-05-15): main.regtech.gold_regtech_reg_migrationinout_population
-- NEW source: main.regtech.gold_regtech_snapshot_reg_regulationinoutdailydata (daily snapshots, LIVE)
-- SSIS parity: truncate/reload snapshot for report date
-- SQL Server equivalent: RegReportDB.dbo.Reg_MigrationInOut_Population WHERE RunDate = @StartDate
--
-- The snapshot table has position-level detail; we derive distinct migration events to match
-- the MigrationInOut_Population grain (one row per CID + regulation change event per day).

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_migrationinout_population
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_migrationinout_population'
AS
SELECT DISTINCT
  src.ReportDate AS RunDate,
  src.CID,
  src.Lei,
  src.RegulationID,
  src.PrevRegulationID,
  src.Migration_Occurred
FROM main.regtech.gold_regtech_snapshot_reg_regulationinoutdailydata src
WHERE src.etr_ymd = '${report_date}'

In [0]:
%sql
-- Reg_RegulationInOutDailyData: from snapshot (LIVE daily data)
-- Original source (STALE since 2023-05-15): main.regtech.gold_regtech_reg_regulationinoutdailydata
-- NEW source: main.regtech.gold_regtech_snapshot_reg_regulationinoutdailydata (daily snapshots, LIVE)
-- NOTE: Must use etr_ymd partition filter to avoid Parquet schema mismatch across older partitions.
-- SSIS parity: truncate/reload snapshot (full daily data for report date)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_regulationinoutdailydata
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_regulationinoutdailydata'
AS
SELECT
  src.*
FROM main.regtech.gold_regtech_snapshot_reg_regulationinoutdailydata src
WHERE src.etr_ymd = '${report_date}'

In [0]:
%sql
-- Reg_Regulation_Movments_Positions: Movement positions combining open, closed, and migration-only rows
-- SSIS parity: Regulation_Movments_Report.dtsx full position logic
-- Dependencies: reg_migrationinout_population (created above), reg_ext_currencypricemaxdatewithsplit (Module 4A)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_regulation_movments_positions
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_regulation_movments_positions'
AS
WITH run_window AS (
  SELECT
    CAST('${report_date}' AS DATE) AS report_date,
    CAST(date_add(CAST('${report_date}' AS DATE), 1) AS TIMESTAMP) AS report_end_ts
),
migration_population_for_run AS (
  SELECT
    src.RunDate,
    src.CID,
    src.Lei,
    src.RegulationID,
    src.Migration_Occurred,
    src.PrevRegulationID
  FROM main.regtech_ops_stg.bi_output_regtechops_reg_migrationinout_population src
  JOIN run_window w ON src.RunDate = w.report_date
),
max_migration AS (
  SELECT
    CID,
    MAX(Migration_Occurred) AS max_migration_occurred
  FROM migration_population_for_run
  GROUP BY CID
),
-- Branch 1: Open positions from Trade.Position
open_positions AS (
  SELECT
    w.report_date AS ReportDate,
    mig.CID,
    mig.RegulationID,
    mig.Migration_Occurred,
    mig.PrevRegulationID,
    tp.PositionID,
    tp.Occurred AS OpenOccurred,
    CAST(NULL AS TIMESTAMP) AS CloseOccurred,
    CAST(tp.IsBuy AS INT) AS IsBuy,
    tp.AmountInUnitsDecimal AS Quantity,
    tp.InitForexRate AS OpenPrice,
    CAST(NULL AS DECIMAL(16, 8)) AS ClosePrice,
    tp.InstrumentID,
    CAST(tp.IsSettled AS INT) AS IsSettled,
    CASE WHEN tp.Occurred >= mm.max_migration_occurred THEN 1 ELSE 0 END AS IsOpenedAfterLastMigration
  FROM migration_population_for_run mig
  JOIN max_migration mm ON mig.CID = mm.CID
  CROSS JOIN run_window w
  JOIN main.trading.silver_etoro_trade_position tp
    ON tp.CID = mig.CID
   AND tp.Occurred < w.report_end_ts
   AND (
        tp.Occurred < mig.Migration_Occurred
        OR (tp.Occurred >= mm.max_migration_occurred AND mig.Migration_Occurred = mm.max_migration_occurred)
   )
),
-- Branch 2: Closed positions from History.Position
closed_positions AS (
  SELECT
    w.report_date AS ReportDate,
    mig.CID,
    mig.RegulationID,
    mig.Migration_Occurred,
    mig.PrevRegulationID,
    hp.PositionID,
    hp.OpenOccurred,
    hp.CloseOccurred,
    CAST(hp.IsBuy AS INT) AS IsBuy,
    hp.AmountInUnitsDecimal AS Quantity,
    hp.InitForexRate AS OpenPrice,
    hp.EndForexRate AS ClosePrice,
    hp.InstrumentID,
    CAST(hp.IsSettled AS INT) AS IsSettled,
    CASE WHEN hp.OpenOccurred >= mm.max_migration_occurred THEN 1 ELSE 0 END AS IsOpenedAfterLastMigration
  FROM migration_population_for_run mig
  JOIN max_migration mm ON mig.CID = mm.CID
  CROSS JOIN run_window w
  JOIN main.trading.bronze_etoro_history_position_datafactory hp
    ON hp.CID = mig.CID
   AND hp.OpenOccurred < w.report_end_ts
   AND (
        (hp.OpenOccurred < mig.Migration_Occurred AND hp.CloseOccurred >= mig.Migration_Occurred)
        OR (hp.OpenOccurred >= mm.max_migration_occurred AND mig.Migration_Occurred = mm.max_migration_occurred)
   )
),
-- Branch 3: Migration-only rows (no matching positions)
migration_only_rows AS (
  SELECT
    w.report_date AS ReportDate,
    mig.CID,
    mig.RegulationID,
    mig.Migration_Occurred,
    mig.PrevRegulationID,
    CAST(NULL AS BIGINT) AS PositionID,
    CAST(NULL AS TIMESTAMP) AS OpenOccurred,
    CAST(NULL AS TIMESTAMP) AS CloseOccurred,
    CAST(NULL AS INT) AS IsBuy,
    CAST(NULL AS DECIMAL(16, 6)) AS Quantity,
    CAST(NULL AS DECIMAL(16, 8)) AS OpenPrice,
    CAST(NULL AS DECIMAL(16, 8)) AS ClosePrice,
    CAST(NULL AS BIGINT) AS InstrumentID,
    CAST(NULL AS INT) AS IsSettled,
    0 AS IsOpenedAfterLastMigration
  FROM migration_population_for_run mig
  CROSS JOIN run_window w
  LEFT JOIN open_positions op
    ON op.CID = mig.CID AND op.RegulationID = mig.RegulationID
   AND op.PrevRegulationID = mig.PrevRegulationID AND op.Migration_Occurred = mig.Migration_Occurred
  LEFT JOIN closed_positions cp
    ON cp.CID = mig.CID AND cp.RegulationID = mig.RegulationID
   AND cp.PrevRegulationID = mig.PrevRegulationID AND cp.Migration_Occurred = mig.Migration_Occurred
  WHERE op.PositionID IS NULL AND cp.PositionID IS NULL
),
movement_base AS (
  SELECT * FROM open_positions
  UNION ALL
  SELECT * FROM closed_positions
  UNION ALL
  SELECT * FROM migration_only_rows
),
movement_enriched AS (
  SELECT
    b.*,
    scd.Symbol,
    scd.IsMifidByESMA,
    scd.IsMifidByFCA,
    CASE
      WHEN b.InstrumentID IS NULL THEN NULL
      WHEN b.IsBuy = 1 THEN CASE WHEN scd.SellCurrencyID = 666 THEN cp.AskSpreaded / 100.0 ELSE cp.AskSpreaded END
      ELSE CASE WHEN scd.SellCurrencyID = 666 THEN cp.BidSpreaded / 100.0 ELSE cp.BidSpreaded END
    END AS EOD_Price,
    CURRENT_TIMESTAMP() AS UpdateDate
  FROM movement_base b
  CROSS JOIN run_window w
  LEFT JOIN main.regtech.gold_regtech_reg_instruments_scd scd
    ON b.InstrumentID = scd.InstrumentID
   AND scd.ValidFrom <= w.report_date
   AND scd.ValidTo > w.report_date
  LEFT JOIN main.regtech_ops_stg.bi_output_regtechops_reg_ext_currencypricemaxdatewithsplit cp
    ON b.InstrumentID = cp.InstrumentID
   AND CAST(cp.OccurredDate AS DATE) = w.report_date
)
SELECT *
FROM movement_enriched

In [0]:
%sql
-- Validation: row counts for Module 5 staging tables
SELECT 'reg_migrationinout_population' AS table_name, COUNT(*) AS row_count FROM main.regtech_ops_stg.bi_output_regtechops_reg_migrationinout_population
UNION ALL SELECT 'reg_regulationinoutdailydata', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_regulationinoutdailydata
UNION ALL SELECT 'reg_regulation_movments_positions', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_regulation_movments_positions